# Evaluation Deep Dive
## Telco Customer Churn Prediction

Going beyond accuracy. Here we look at how the best model actually performs —
where it gets things right, where it fails, and what that means for the business.

Sections:
1. Load model and data
2. Confusion matrix
3. ROC and Precision-Recall curves
4. Feature importance
5. Learning curves
6. Error analysis — what kinds of customers does it get wrong?
7. Final threshold recommendation

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
warnings.filterwarnings("ignore")

from sklearn.metrics import classification_report, f1_score, roc_auc_score

from src.preprocessing import load_data, clean_data, split_data
from src.features import engineer_features
from src.evaluate import (
    plot_confusion_matrix, plot_roc_curves, plot_pr_curves,
    plot_feature_importance, plot_learning_curves,
    plot_threshold_analysis, get_misclassified,
)

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13, "axes.labelsize": 11})

## 1. Load Model and Data

In [ ]:
RAW_PATH = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = engineer_features(clean_data(load_data(RAW_PATH)))
X_train, X_test, y_train, y_test = split_data(df)

preprocessor = joblib.load("../models/preprocessor.joblib")
X_train_proc  = preprocessor.transform(X_train)
X_test_proc   = preprocessor.transform(X_test)

best_model = joblib.load("../models/best_model.joblib")

with open("../models/model_meta.json") as f:
    meta = json.load(f)

print(f"Best model: {meta['model_name']}")
print(f"Optimal threshold: {meta['optimal_threshold']}")
print(f"Test F1 (at optimal threshold): {meta['tuned_f1_test']}")
print(f"Test ROC-AUC: {meta['tuned_roc_auc_test']}")
print(f"CV F1: {meta['cv_f1_mean']} +/- {meta['cv_f1_std']}")

In [ ]:
# quick summary at both default and optimal threshold
y_prob = best_model.predict_proba(X_test_proc)[:, 1]
optimal_t = meta["optimal_threshold"]

for label, t in [("Default (0.50)", 0.50), (f"Optimal ({optimal_t})", optimal_t)]:
    y_pred_t = (y_prob >= t).astype(int)
    print(f"\n{label}")
    print(f"  F1:      {f1_score(y_test, y_pred_t):.4f}")
    print(f"  ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

## 2. Confusion Matrix

The confusion matrix breaks down exactly where the model succeeds and fails.

For churn prediction, **false negatives** (missed churners) are the most costly error —
a missed churner means lost revenue with no chance to intervene.
False positives (false alarms) waste retention budget on customers who weren't leaving anyway.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# at default threshold
y_pred_default = (y_prob >= 0.50).astype(int)
plot_confusion_matrix(y_test, y_pred_default, model_name=f"{meta['model_name']} — threshold 0.50", ax=axes[0])

# at optimal threshold
y_pred_optimal = (y_prob >= optimal_t).astype(int)
plot_confusion_matrix(y_test, y_pred_optimal, model_name=f"{meta['model_name']} — threshold {optimal_t}", ax=axes[1])

plt.tight_layout()
plt.savefig("../reports/fig_15_confusion_matrices.png", bbox_inches="tight")
plt.show()

In [ ]:
# detailed classification report at optimal threshold
print(f"Classification Report at threshold {optimal_t}:\n")
print(classification_report(y_test, y_pred_optimal, target_names=["No Churn", "Churn"]))

total_churners = y_test.sum()
caught   = ((y_pred_optimal == 1) & (y_test == 1)).sum()
missed   = ((y_pred_optimal == 0) & (y_test == 1)).sum()
false_alarm = ((y_pred_optimal == 1) & (y_test == 0)).sum()

print(f"Out of {total_churners} actual churners:")
print(f"  Caught:      {caught} ({caught/total_churners:.1%})")
print(f"  Missed:      {missed} ({missed/total_churners:.1%})")
print(f"  False alarms (non-churners flagged): {false_alarm}")

## 3. ROC and Precision-Recall Curves

Both curves compare models visually. A few things to note:

- **ROC-AUC** measures the model's ability to rank churners higher than non-churners across all thresholds.
  An AUC of 0.5 is random, 1.0 is perfect.
- **PR curves** are more informative for imbalanced datasets because they focus specifically
  on the minority class (churners). The baseline on a PR curve is the churn rate itself (~0.27),
  not 0.5 like in ROC.

In [ ]:
# load top 3 models to compare curves
tuned_xgb = joblib.load("../models/tuned_models/xgboost_tuned.joblib")
tuned_rf  = joblib.load("../models/tuned_models/random_forest_tuned.joblib")
base_xgb  = joblib.load("../models/baseline_models/xgboost.joblib")

compare_models = {
    "XGBoost (tuned)":        tuned_xgb,
    "Random Forest (tuned)":  tuned_rf,
    "XGBoost (baseline)":     base_xgb,
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_roc_curves(compare_models, X_test_proc, y_test, ax=axes[0])
plot_pr_curves( compare_models, X_test_proc, y_test, ax=axes[1])

plt.suptitle("Model Comparison — ROC and PR Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/fig_16_roc_pr_curves.png", bbox_inches="tight")
plt.show()

## 4. Feature Importance

Which features are driving the predictions the most?

In [ ]:
from src.features import get_feature_names

numerical_cols = [
    "tenure", "MonthlyCharges", "TotalCharges",
    "monthly_tenure_ratio", "total_services", "avg_monthly_spend",
    "contract_risk",
]
categorical_cols = [
    "gender", "SeniorCitizen", "Partner", "Dependents",
    "PhoneService", "MultipleLines", "InternetService",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaperlessBilling", "PaymentMethod",
    "tenure_group",
]

try:
    feature_names = get_feature_names(preprocessor, numerical_cols, categorical_cols)
except Exception:
    # fallback: use index numbers
    feature_names = [f"feature_{i}" for i in range(X_train_proc.shape[1])]

fig, ax = plt.subplots(figsize=(11, 8))
plot_feature_importance(best_model, feature_names, top_n=20, ax=ax)
plt.tight_layout()
plt.savefig("../reports/fig_17_feature_importance.png", bbox_inches="tight")
plt.show()

In [ ]:
# show top 10 as a ranked table
importances = pd.Series(best_model.feature_importances_, index=feature_names)
top10 = importances.sort_values(ascending=False).head(10).reset_index()
top10.columns = ["Feature", "Importance"]
top10["Importance"] = top10["Importance"].round(4)
display(top10)

## 5. Learning Curves

Learning curves show how the model's training and validation scores change as we add more training data.

- If the training score is much higher than validation: the model is overfitting.
- If both scores are low and converging: the model is underfitting.
- If the validation score is still rising at the far right: more data would likely help.

In [ ]:
from sklearn.model_selection import StratifiedKFold

fig, ax = plt.subplots(figsize=(10, 5))
plot_learning_curves(
    best_model, X_train_proc, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="f1",
    ax=ax,
)
plt.tight_layout()
plt.savefig("../reports/fig_18_learning_curves.png", bbox_inches="tight")
plt.show()

## 6. Error Analysis

Looking at the specific customers the model gets wrong tells us more than any metric.

- **False negatives (missed churners):** These are the most costly. Who are they?
- **False positives (false alarms):** Why did the model flag these customers incorrectly?

In [ ]:
fn_df, fp_df = get_misclassified(
    best_model, X_test_proc, y_test, X_test,
    threshold=optimal_t, n=10
)

print(f"Worst false negatives (churners we missed) — {len(fn_df)} shown")
print(f"Model was most confident these customers would NOT churn, but they did.\n")
display(fn_df[["tenure", "Contract", "MonthlyCharges", "InternetService",
               "TechSupport", "churn_prob", "actual", "predicted"]].head(5))

In [ ]:
print(f"Worst false positives (false alarms) — {len(fp_df)} shown")
print(f"Model was most confident these customers would churn, but they stayed.\n")
display(fp_df[["tenure", "Contract", "MonthlyCharges", "InternetService",
               "TechSupport", "churn_prob", "actual", "predicted"]].head(5))

In [ ]:
# compare average feature values: missed churners vs caught churners vs false alarms
y_pred_all = (y_prob >= optimal_t).astype(int)
X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

groups = {
    "Caught churners (TP)":  X_test_reset[(y_test_reset == 1) & (y_pred_all == 1)],
    "Missed churners (FN)":  X_test_reset[(y_test_reset == 1) & (y_pred_all == 0)],
    "False alarms (FP)":     X_test_reset[(y_test_reset == 0) & (y_pred_all == 1)],
}

num_cols_simple = ["tenure", "MonthlyCharges", "TotalCharges", "total_services"]
summary = pd.DataFrame({name: grp[num_cols_simple].mean() for name, grp in groups.items()}).T
print("Average feature values by prediction group:")
display(summary.round(2))

Looking at these groups gives a clearer picture of when the model struggles:

- Missed churners tend to have longer tenure — the model relies heavily on tenure as a
  safety signal, so long-tenure customers who eventually churn fool it.
- False alarms often have month-to-month contracts and high monthly charges — exactly the
  profile the model learned to flag — but these particular customers are satisfied enough to stay.

## 7. Final Threshold Recommendation

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
_, opt_t = plot_threshold_analysis(y_test, y_prob, ax=ax)
plt.tight_layout()
plt.savefig("../reports/fig_19_threshold_recommendation.png", bbox_inches="tight")
plt.show()

print(f"\nRecommended threshold: {opt_t:.2f}")
print()
print(f"{'Threshold':>10}  {'Precision':>10}  {'Recall':>8}  {'F1':>8}  {'Interpretation'}")
print("-" * 75)

from sklearn.metrics import precision_score, recall_score

for t, interp in [(0.30, "High recall — catch almost all churners, more false alarms"),
                  (0.40, "Balanced toward recall"),
                  (0.50, "Default — balanced but misses more churners"),
                  (opt_t, "Optimal F1"),
                  (0.60, "High precision — fewer false alarms, misses more churners")]:
    y_p = (y_prob >= t).astype(int)
    p = precision_score(y_test, y_p, zero_division=0)
    r = recall_score(y_test, y_p, zero_division=0)
    f = f1_score(y_test, y_p, zero_division=0)
    marker = " <--" if abs(t - opt_t) < 0.01 else ""
    print(f"{t:>10.2f}  {p:>10.4f}  {r:>8.4f}  {f:>8.4f}  {interp}{marker}")

print(f"\nFinal recommendation: use threshold {opt_t:.2f}")
print("This maximises F1 — the best balance of catching churners without flooding")
print("the retention team with too many false alarms.")